## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. The first cell installs what's needed,
downloads the camp toolbox, and downloads the data — just press ▶ and wait for
the green **✅ Setup complete**, then run the rest of the notebook top to bottom.

It also offers to connect your Google Drive so your figures are *saved* for your
poster (recommended). If you skip that, the notebook still works — your figures
just won't persist after you close Colab.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os

print("1/3  installing libraries ...")
get_ipython().system('pip install -q "mne==1.10.1" gdown')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

print("3/3  downloading the data (~470 MB, first time only) ...")
import gdown
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/synapse_preprocessed.pkl"):
    gdown.download(id="1Z-NENlKMjL-kL-N46lQ8QA1AbGM7bJHY",
                   output="data/synapse_preprocessed.pkl", quiet=False)
os.environ["CAMP_DATA_PATH"] = "data/synapse_preprocessed.pkl"

# Save figures to your own Drive so they persist for your poster (recommended).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
    where = "Drive > DecodingBrain_outputs"
except Exception:
    os.environ["CAMP_OUTPUT_DIR"] = "outputs"
    where = "a temporary 'outputs' folder (download anything you want to keep!)"
print(f"\n\u2705 Setup complete. Figures will be saved to {where}.")


# Week 2 · Day 7 — Publication-Quality Figures

A great result hidden in an ugly chart won't convince anyone. Scientists spend
real effort making figures that are **clear, honest, and accessible**. Today you
learn the craft: `seaborn` styling, a **colorblind-safe** palette, clear labels,
and multi-panel layouts — the same standards used in the SYNAPSE paper.

### By the end of this notebook you will be able to
1. Use `seaborn` for clean statistical plots
2. Apply the camp's consistent colorblind-safe colors
3. Show the data *and* its spread honestly (box + individual points)
4. Build a multi-panel figure and label it like a journal figure

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import camp_utils as cu

# load the table you built in Notebook 05
features = pd.read_csv(cu.save_path("features_table.csv"))
print("Loaded features:", features.shape)
features.head(3)

## 1. Why color matters
About 1 in 12 men has some red-green colorblindness. We use the **Wong palette**
— orange for EXP, blue for CTRL — which everyone can tell apart. These are stored
in `camp_utils` so every figure in the camp matches.

In [ ]:
palette = {"EXP": cu.EXP_COLOR, "CTRL": cu.CTRL_COLOR}
print("EXP  =", cu.EXP_COLOR, "(orange)")
print("CTRL =", cu.CTRL_COLOR, "(blue)")

sns.set_theme(style="ticks", context="notebook")   # clean seaborn defaults

## 2. A good single panel: box + strip
A boxplot shows the summary (median, quartiles); a stripplot shows every actual
subject. Showing both is honest — the reader sees the spread, not just a bar.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
# palette is keyed by the x categories ("EXP"/"CTRL"), so each box gets its color
sns.boxplot(data=features, x="group", y="let_gamma",
            palette=palette, width=0.5, fliersize=0, ax=ax)
sns.stripplot(data=features, x="group", y="let_gamma",
              color="black", size=5, alpha=0.6, jitter=0.15, ax=ax)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("")
ax.set_ylabel("LET gamma power change (dB)")
ax.set_title("Gamma response during Listening Effort")
sns.despine()
plt.tight_layout()
plt.show()

### ✏️ Your turn #1 — style a panel
Make the same box+strip panel for a **different feature**. Pick any column from
`features` (try `let_alpha`, `ast_theta`, or `let_n1_amp`). Give it a clear
y-axis label and title.

In [ ]:
my_feature = "let_alpha"   # TODO: try a few

# TODO: copy the box+strip pattern above, swapping y=my_feature.
#       Remember: clear ylabel, clear title, sns.despine().

## 3. Labels make or break a figure
Rules we follow in this camp (from the SYNAPSE figure style):
- Every axis has a **label with units** (dB, µV, ms).
- Font size **≥ 12** so it's readable when shrunk.
- **No chart-junk** — no 3D, no rainbow gradients.
- A short, specific **title** that states what's shown.

Here's a helper that turns our terse feature names into nice labels.

In [ ]:
NICE_NAMES = {
    "let_gamma": "LET · Gamma (dB)", "let_alpha": "LET · Alpha (dB)",
    "let_beta": "LET · Beta (dB)", "ast_theta": "AST · Theta (dB)",
    "ast_delta": "AST · Delta (dB)", "let_n1_amp": "LET · N1 amplitude (µV)",
    "let_p2_amp": "LET · P2 amplitude (µV)",
}
def nice(name):
    return NICE_NAMES.get(name, name)

print("let_gamma  ->", nice("let_gamma"))

## 4. A multi-panel figure
Journal figures usually have several panels (a, b, c…) in one image. We'll show
four features side by side. The pattern: make a grid of axes, loop, draw one
panel per feature, and label each with a letter.

In [ ]:
panel_features = ["let_gamma", "let_alpha", "let_beta", "ast_theta"]
panel_letters = ["a", "b", "c", "d"]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, feat, letter in zip(axes, panel_features, panel_letters):
    sns.boxplot(data=features, x="group", y=feat,
                palette=palette, width=0.5, fliersize=0, ax=ax)
    sns.stripplot(data=features, x="group", y=feat,
                  color="black", size=4, alpha=0.6, jitter=0.15, ax=ax)
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel(""); ax.set_ylabel(nice(feat))
    # panel letter in the top-left corner
    ax.set_title(letter, loc="left", fontweight="bold", fontsize=14)
    sns.despine(ax=ax)

fig.suptitle("EXP vs CTRL across four features", y=1.03, fontsize=14)
plt.tight_layout()
plt.savefig(cu.save_path("multipanel_features.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved at 300 DPI (print quality) to outputs/multipanel_features.png")

### ✏️ Your turn #2 — design your own multi-panel
Build a **2×2** multi-panel figure (4 panels) using any 4 features you find
interesting. Requirements:
- colorblind-safe palette (use `palette`)
- labelled y-axes with units (use `nice()` or write your own)
- panel letters a, b, c, d
- saved to `outputs/` at `dpi=300`

*Hint:* `plt.subplots(2, 2, figsize=(9, 8))` gives a 2×2 grid; loop over
`axes.flat`.

In [ ]:
# TODO: your 2x2 multi-panel figure here

## 🎯 Wrap-up
You can now make figures that look like they belong in a journal: honest spread,
accessible color, clear labels, multiple panels. You'll reuse these skills for
your **poster** in Week 4.

➡️ **Next:** Notebook 07 — but is any of this *statistically real*? Time for our
first hypothesis test.